# Week 01 Supplementary Notes: Correlation and Data Augmentation

Companion notebook for the supplementary notes, *AI Principles and Practice*, COMP-0484.

Every figure and every number in the notes is reproduced here. Nothing is hidden in a helper
module, so you can change one thing at a time and watch what happens.

**How to use this.** Run it top to bottom once. Then go back to the cells marked
`TRY THIS` and change the parameter they name. The point of the notebook is the second pass,
not the first.

Sections 1 to 5 run on a CPU runtime in under a minute. Section 6 downloads models from
Hugging Face and is happier with a GPU. Section 7 runs on your own machine only.

## 0. Setup

In [ ]:
# On Colab, run this once. Locally, install into your environment instead.
%pip install -q dcor imbalanced-learn

In [ ]:
import random, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import dcor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
rng = np.random.default_rng(SEED)

pd.set_option('display.width', 110)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.color': '#e3e7ec', 'axes.axisbelow': True})

print('numpy', np.__version__, '| pandas', pd.__version__)

### The running example

The same synthetic customers table as the lecture, with the columns the slides referred to but
did not build: `monthly_charge`, which is monotonically related to income, and a `churn` target.
Income carries a small number of extreme values on purpose.

In [ ]:
def make_customers(n=600, seed=42):
    rng = np.random.default_rng(seed)
    income = rng.lognormal(10.6, 0.45, n).round()

    # a handful of genuinely extreme earners, the outliers the lecture flagged
    idx = rng.choice(n, 4, replace=False)
    income[idx] *= rng.uniform(8, 22, 4)

    # charge rises with income, but through the RANK, not the raw value
    pct = stats.rankdata(income) / n
    charge = 21 + 55 * pct + rng.normal(0, 4.5, n)

    df = pd.DataFrame({
        'age':          rng.normal(40, 12, n).round().clip(18, 80),
        'income':       income.round(),
        'tenure':       rng.integers(0, 72, n),
        'monthly_charge': charge.round(2),
        'plan':         rng.choice(['Basic', 'Standard', 'Premium'], n),
        'region':       rng.choice(['North', 'South', 'East', 'West'], n),
        'satisfaction': rng.choice(['Low', 'Medium', 'High'], n, p=[.25, .45, .30]),
    })

    # churn: more likely when satisfaction is low and tenure is short
    logit = (-2.2
             + 1.5 * (df.satisfaction == 'Low')
             - 0.02 * df.tenure
             + rng.normal(0, 0.6, n))
    df['churn'] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)

    # deliberate gaps, as in the lecture
    df.loc[rng.random(n) < 0.088, 'income'] = np.nan
    df.loc[rng.random(n) < 0.050, 'monthly_charge'] = np.nan
    return df

df = make_customers()
print(df.shape, '| churn rate:', round(df.churn.mean(), 3))
df.head()

---
## 1. Two coefficients that do not agree

This is the result from the lecture. Same two columns, two answers.

In [ ]:
sub = df[['income', 'monthly_charge']].dropna()
r,   p_r = stats.pearsonr(sub.income, sub.monthly_charge)
rho, p_s = stats.spearmanr(sub.income, sub.monthly_charge)

print(f'Pearson  r   = {r:.3f}   (p = {p_r:.1e})')
print(f'Spearman rho = {rho:.3f}   (p = {p_s:.1e})')
print()
print('Pearson on log(income) =',
      round(stats.pearsonr(np.log(sub.income), sub.monthly_charge)[0], 3))
print('Spearman is unchanged by the log, because ranks do not move.')

**TRY THIS.** In `make_customers`, change `rng.uniform(8, 22, 4)` to `rng.uniform(1, 1, 4)`,
which removes the outliers entirely, and rerun. Pearson and Spearman should converge.
Four rows out of six hundred were doing all the damage.

## 2. The full association report

Chatterjee's coefficient and distance correlation, added to the three classical ones.

In [ ]:
def chatterjee_xi(x, y, rng=np.random.default_rng(0)):
    """Chatterjee (2021). Asymmetric: how nearly is y a function of x?"""
    x, y = np.asarray(x, float), np.asarray(y, float)
    n = len(x)
    order = np.argsort(x + rng.random(n) * 1e-9)     # random tie-breaking
    r = stats.rankdata(y[order], method='max')
    return 1.0 - 3.0 * np.abs(np.diff(r)).sum() / (n**2 - 1)


def association_report(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)             # pairwise deletion, stated openly
    x, y = x[ok], y[ok]
    r = stats.pearsonr(x, y)[0]
    z, se = np.arctanh(r), 1 / np.sqrt(len(x) - 3)
    return pd.Series({
        'n':            len(x),
        'pearson_r':    round(r, 3),
        'pearson_lo':   round(np.tanh(z - 1.96 * se), 3),
        'pearson_hi':   round(np.tanh(z + 1.96 * se), 3),
        'spearman_rho': round(stats.spearmanr(x, y)[0], 3),
        'kendall_tau':  round(stats.kendalltau(x, y)[0], 3),
        'distance_cor': round(dcor.distance_correlation(x, y), 3),
        'xi_x_to_y':    round(chatterjee_xi(x, y), 3),
        'xi_y_to_x':    round(chatterjee_xi(y, x), 3),
    })

association_report(df.income, df.monthly_charge)

### Six shapes, five coefficients (Figure 1 in the notes)

In [ ]:
def six_patterns(n=300, seed=7):
    g = np.random.default_rng(seed)
    out = {}
    x = g.uniform(-3, 3, n);  out['Linear']        = (x, 1.2 * x + g.normal(0, 1.0, n))
    x = g.uniform(0, 3, n);   out['Curved']        = (x, np.exp(x) + g.normal(0, 1.0, n))
    x = g.uniform(-3, 3, n);  out['U shape']       = (x, x**2 + g.normal(0, 0.8, n))
    x = g.uniform(0, 4*np.pi, n); out['Periodic']  = (x, np.sin(x) + g.normal(0, .18, n))
    x = np.r_[g.normal(0, 1, n-6), g.normal(9, .4, 6)]
    y = np.r_[g.normal(0, 1, n-6), g.normal(9, .4, 6)]
    out['Outliers only'] = (x, y)
    x = g.uniform(0, 4, n);   out['Fan']           = (x, g.normal(0, 0.25 + 0.75 * x))
    return out


cases = six_patterns()
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
rows = {}
for ax, (name, (x, y)) in zip(axes.ravel(), cases.items()):
    ax.scatter(x, y, s=7, color='#7b9bd1', alpha=.75, linewidths=0)
    ax.set_title(name, weight='bold', color='#2f4a68')
    ax.set_xticks([]); ax.set_yticks([])
    rows[name] = association_report(x, y)
plt.tight_layout(); plt.show()

pd.DataFrame(rows).T[['pearson_r', 'spearman_rho', 'kendall_tau',
                      'distance_cor', 'xi_x_to_y', 'xi_y_to_x']]

Read the table across, not down.

- **U shape** and **Periodic** are near-deterministic relationships. Pearson, Spearman *and*
  Kendall all report roughly zero. Distance correlation and Chatterjee's xi find them.
- **Outliers only** has no internal relationship. Pearson reports a healthy positive number.
  Notice that distance correlation is also fooled: it is sensitive to *any* dependence, and six
  points in their own cluster is a real dependence. Sensitivity and robustness are different
  properties.
- **Fan** has no trend in the mean, only in the spread. Every classical measure says zero and is
  technically correct while missing something real.
- Compare `xi_x_to_y` against `xi_y_to_x` for the U shape. y is a function of x; x is not a
  function of y. The asymmetry is the point.

## 3. Simpson's paradox

The correlation within every group has the opposite sign to the correlation across all groups.

In [ ]:
g = np.random.default_rng(3)
parts = []
for name, t_mid, c_mid in [('Basic', 11, 22), ('Standard', 29, 40), ('Premium', 48, 62)]:
    t = np.clip(g.normal(t_mid, 4.5, 150), 0, 71)
    c = c_mid - 0.55 * (t - t_mid) + g.normal(0, 2.2, 150)
    parts.append(pd.DataFrame({'plan': name, 'tenure': t, 'charge': c}))
sim = pd.concat(parts, ignore_index=True)

for name, grp in sim.groupby('plan'):
    print(f'{name:10s} r = {stats.pearsonr(grp.tenure, grp.charge)[0]:+.2f}')
print(f'{"POOLED":10s} r = {stats.pearsonr(sim.tenure, sim.charge)[0]:+.2f}   <- sign flipped')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, grp in sim.groupby('plan'):
    for a in ax:
        a.scatter(grp.tenure, grp.charge, s=9, alpha=.65, linewidths=0, label=name)
    s, i = np.polyfit(grp.tenure, grp.charge, 1)
    xs = np.linspace(grp.tenure.min(), grp.tenure.max(), 10)
    ax[0].plot(xs, s * xs + i, lw=2)
s, i = np.polyfit(sim.tenure, sim.charge, 1)
xs = np.linspace(sim.tenure.min(), sim.tenure.max(), 10)
ax[1].plot(xs, s * xs + i, lw=2.5, ls='--', color='#2f4a68')
ax[0].set_title('Within each plan'); ax[1].set_title('Pooled')
ax[0].legend(frameon=False)
for a in ax: a.set_xlabel('tenure (months)'); a.set_ylabel('monthly charge')
plt.tight_layout(); plt.show()

**TRY THIS.** Run the same check on your own project dataset: compute the correlation of your
two most important numeric columns overall, then within each level of every categorical column
you have. Exercise 3 in the notes.

---
## 4. Resampling: what SMOTE and its variants actually do

Figure 3 in the notes.

In [ ]:
from sklearn.datasets import make_classification
from imblearn.over_sampling import (RandomOverSampler, SMOTE,
                                    BorderlineSMOTE, ADASYN)

X, y = make_classification(n_samples=900, n_features=2, n_redundant=0,
                           n_informative=2, n_clusters_per_class=1,
                           weights=[0.93, 0.07], flip_y=0.02,
                           class_sep=1.0, random_state=4)

samplers = [('Original', None),
            ('RandomOverSampler', RandomOverSampler(random_state=0)),
            ('SMOTE', SMOTE(random_state=0)),
            ('BorderlineSMOTE', BorderlineSMOTE(random_state=0)),
            ('ADASYN', ADASYN(random_state=0))]

fig, axes = plt.subplots(1, 5, figsize=(17, 3.6))
for ax, (name, samp) in zip(axes, samplers):
    Xs, ys = (X, y) if samp is None else samp.fit_resample(X, y)
    ax.scatter(*Xs[ys == 0].T, s=5, color='#7b9bd1', alpha=.45, linewidths=0)
    ax.scatter(*Xs[ys == 1].T, s=7, color='#b5432a', alpha=.75, linewidths=0)
    ax.set_title(f'{name}\nminority n = {(ys == 1).sum()}', fontsize=9, weight='bold')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

Look at the long synthetic bridges SMOTE draws through the majority cloud. They come from a
few minority points sitting deep inside majority territory, most likely mislabelled. SMOTE
interpolates towards them and invents confidently labelled minority examples in a region where
the minority class does not live. Borderline-SMOTE refuses to synthesise from interior points.
ADASYN deliberately emphasises hard examples and so makes it worse.

## 5. The leakage demo

This is the most important cell in the notebook.

The data below is **pure noise** with **coin-flip labels**. There is nothing to learn.
Any score above 0.5 is manufactured.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline   # NOT sklearn's Pipeline

g = np.random.default_rng(0)
n, p = 500, 25
Xn = g.normal(size=(n, p))                      # pure noise
yn = (g.random(n) < 0.12).astype(int)           # labels unrelated to Xn

cv = StratifiedKFold(5, shuffle=True, random_state=0)
def forest():
    return RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)

# WRONG: resample everything, then cross-validate
Xr, yr = SMOTE(random_state=0).fit_resample(Xn, yn)
leaked = cross_val_score(forest(), Xr, yr, cv=cv, scoring='roc_auc').mean()

# RIGHT: resampling lives inside the pipeline, refit on each training fold
pipe = ImbPipeline([('smote', SMOTE(random_state=0)), ('clf', forest())])
honest = cross_val_score(pipe, Xn, yn, cv=cv, scoring='roc_auc').mean()

print(f'SMOTE before the split, then CV : {leaked:.3f}   <- fiction')
print(f'SMOTE inside an imblearn Pipeline: {honest:.3f}   <- chance, as it must be')
print(f'Chance                          : 0.500')

**Why.** Every synthetic minority point is a weighted average of two real minority points. When
SMOTE runs before the split, a synthetic point can land in the validation fold while both of its
parents sit in the training fold. The model has already memorised the ingredients.

**The same argument applies to every other augmentation.** Rotated copies of training photographs
in the test set. Back-translated paraphrases of training sentences in the test set. A CTGAN fitted
on the full table before splitting. Split first, augment the training portion only.

**TRY THIS.** Change `0.12` to `0.30` and rerun. The inflation shrinks, because less synthesis is
needed to balance the classes. Then try `0.05`.

## 6. Is resampling even the right answer?

On the real customers table this time, with an honest held-out test set. Four approaches, one
model, one metric. Note that we report average precision (the area under the precision-recall
curve) rather than accuracy, because accuracy is what made the imbalance look like a problem in
the first place.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.metrics import average_precision_score

num = ['age', 'income', 'tenure', 'monthly_charge']
cat = ['plan', 'region', 'satisfaction']
X, y = df[num + cat], df.churn

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.25, stratify=y, random_state=0)

def prep():
    return ColumnTransformer([
        ('num', SkPipeline([('impute', SimpleImputer(strategy='median')),
                            ('scale',  StandardScaler())]), num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])

candidates = {
    'no treatment':      ImbPipeline([('prep', prep()), ('clf', forest())]),
    'class_weight':      ImbPipeline([('prep', prep()),
                                      ('clf', RandomForestClassifier(
                                          n_estimators=200, class_weight='balanced',
                                          random_state=0, n_jobs=-1))]),
    'SMOTE (full)':      ImbPipeline([('prep', prep()),
                                      ('smote', SMOTE(random_state=0)),
                                      ('clf', forest())]),
    'SMOTE (ratio 0.4)': ImbPipeline([('prep', prep()),
                                      ('smote', SMOTE(sampling_strategy=0.4, random_state=0)),
                                      ('clf', forest())]),
}

results = []
for name, model in candidates.items():
    cv_ap = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='average_precision').mean()
    model.fit(X_tr, y_tr)
    test_ap = average_precision_score(y_te, model.predict_proba(X_te)[:, 1])
    results.append({'approach': name, 'cv_AP': round(cv_ap, 3), 'test_AP': round(test_ap, 3)})

print('test set base rate (the floor for AP):', round(y_te.mean(), 3))
pd.DataFrame(results)

The test set was never resampled and is still imbalanced, which is the only way these numbers
mean anything operationally. Compare each score against the base rate printed above, which is
what a model guessing at random would score.

Look at the two columns against each other before you read the rows. On the default seed, SMOTE
posts the best cross-validated score and the *worst* test score. Resampling made the model look
better at the thing it was tuned on and no better at the thing it was for. Class weighting,
which invents no data and carries no leakage risk, comes out ahead on the held-out set.

This will not happen on every dataset. It happens often enough that class weighting is the first
thing to try, not the fallback.

**TRY THIS.** Change `random_state=0` in the `train_test_split` to a few other values and watch
how much the ordering moves. A single split is a noisy way to choose between four approaches.

---
## 7. Text augmentation with Hugging Face

This section downloads models. It works on a CPU runtime but is slow; on Colab, switch to a GPU
under **Runtime > Change runtime type** and set `device=0` below.

In [ ]:
%pip install -q transformers sentencepiece sacremoses

In [ ]:
from transformers import pipeline

DEVICE = -1          # set to 0 if you have a GPU runtime

to_fr = pipeline('translation', model='Helsinki-NLP/opus-mt-en-fr', device=DEVICE)
to_en = pipeline('translation', model='Helsinki-NLP/opus-mt-fr-en', device=DEVICE)

def back_translate(texts, batch_size=8):
    fr = [t['translation_text'] for t in to_fr(texts, batch_size=batch_size)]
    return [t['translation_text'] for t in to_en(fr, batch_size=batch_size)]

originals = [
    'The support agent resolved my billing issue within a few minutes.',
    'I have been waiting three weeks for a refund and nobody has replied.',
    'Coverage is patchy at home but the price is fair enough.',
]
for o, a in zip(originals, back_translate(originals)):
    print('orig:', o)
    print('aug :', a, '\n')

**Read the output, do not just admire it.** You are checking rule one from the notes: did the
label survive the round trip? A complaint that comes back as a mild observation is a
label-flipping augmentation and must be discarded. This inspection step is the whole exercise,
and it is the step people skip.

In [ ]:
fill = pipeline('fill-mask', model='distilroberta-base', device=DEVICE)

s = 'The delivery was <mask> and the packaging was damaged.'
for cand in fill(s, top_k=5):
    print(round(cand['score'], 3), repr(cand['token_str'].strip()), '|', cand['sequence'])

The model proposes candidates with opposite implications for a sentiment label. Contextual
substitution solves the WordNet synonym problem and introduces a new one. The fix is to
constrain the candidate set, to condition on the label, or to run a trained classifier over the
augmented text and drop anything whose predicted label changed.

---
## 8. Local generation with Ollama

**This section will not run on Colab.** It needs Ollama installed on your own machine, which is
the entire point: your data never leaves it. Install from [ollama.com](https://ollama.com), then:

```bash
ollama pull llama3.2:3b
pip install ollama
```

Then run the cell below locally.

In [ ]:
# LOCAL ONLY. Requires a running Ollama daemon on localhost:11434.
import json

SCHEMA = {
    'type': 'object',
    'properties': {
        'age':          {'type': 'integer'},
        'income':       {'type': 'integer'},
        'tenure':       {'type': 'integer'},
        'plan':         {'type': 'string', 'enum': ['Basic', 'Standard', 'Premium']},
        'satisfaction': {'type': 'string', 'enum': ['Low', 'Medium', 'High']},
        'churn':        {'type': 'integer', 'enum': [0, 1]},
    },
    'required': ['age', 'income', 'tenure', 'plan', 'satisfaction', 'churn'],
}

def synth_row(seed_rows, model='llama3.2:3b'):
    import ollama
    prompt = (
        'You are generating additional plausible rows for a telecom customer table.\n'
        'Here are real examples:\n' + seed_rows + '\n'
        'Produce ONE new row with churn = 1. Ages 18-80, income in euro, tenure in months. '
        'It must be realistic and must not copy any example.')
    r = ollama.chat(model=model,
                    messages=[{'role': 'user', 'content': prompt}],
                    format=SCHEMA,                      # constrained decoding -> valid JSON
                    options={'temperature': 0.9, 'seed': 0})
    return json.loads(r['message']['content'])

# NOTE: seed rows come from the TRAINING split only. Same leakage rule as section 5.
# seed = X_tr.join(y_tr)[lambda d: d.churn == 1].sample(5, random_state=0).to_csv(index=False)
# new_rows = pd.DataFrame([synth_row(seed) for _ in range(20)])
# new_rows

Before any of that enters a training set: sample twenty rows and read them, check for
near-duplicates, compare each synthetic column's distribution against the real one, and train a
classifier to tell real from synthetic. If that classifier does well, your synthetic data does
not resemble the real data and should not be used. Exercise 7 in the notes.

---
## Where to go next

The seven exercises at the end of the supplementary notes all start from cells in this notebook.
Exercises 1, 4 and 5 are the ones worth doing before the lab.

When you build your own pipeline for the lab deliverable, reuse the `ImbPipeline` pattern from
section 6. It is already leakage-safe, and it is what Week 2 trains models on top of.